In [1]:
!pip install -q datasets transformers huggingface_hub accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


In [2]:
import torch
torch.cuda.is_available()

True

In [3]:
# Chinese dataset
import pandas as pd
from datasets import Dataset, DatasetDict

# 1. Fetch raw data splits straight from GitHub
train_url = "https://raw.githubusercontent.com/wangy8989/Chinese-Financial-News-Sentiment-Analysis/main/Train_Data.csv"

df_train = pd.read_csv(train_url)

# Drop the blank, null rows
df_train = df_train.dropna(subset=["text"])

# Explicitly cast the text column to clean Python string formats
df_train["text"] = df_train["text"].astype(str)

# Convert into Hugging Face native datasets
full_dataset = Dataset.from_pandas(df_train, preserve_index=False)

# Split into distinct train and test splits (90% train, 10% validation)

split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Dataset is good and splited")
print(f"Training samples: {len(train_dataset)} | Validation samples: {len(test_dataset)}")

Dataset is good and splited
Training samples: 4499 | Validation samples: 500


In [4]:
# Tokenizer
from transformers import AutoTokenizer

model_checkpoint = "FacebookAI/xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [5]:
# Tokenizer Preprocessing Funciton
def preprocess_function(examples):
    # Tokenize text inputs
    tokenized = tokenizer(examples["text"], truncation=True, max_length=128)

    # Map the column with the expected value of the model
    tokenized["labels"] = examples["negative"]

    return tokenized

tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_test = test_dataset.map(preprocess_function, batched=True)

print(f'\nTokenization Complete...')


Map:   0%|          | 0/4499 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]


Tokenization Complete...


In [6]:
import numpy as np
import evaluate
from transformers import DataCollatorWithPadding

# Strips out token_type_ids that are incompatible with RoBERTa models
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

load_accuracy = evaluate.load("accuracy")
load_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = load_accuracy.compute(predictions=predictions, references=labels)["accuracy"]
    # Using macro averaging ensures balanced accountability across both sentiment weights
    f1 = load_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]

    return {"accuracy": accuracy, "f1": f1}

In [7]:
from transformers import AutoModelForSequenceClassification

# 2 labels matching Binary Classification (0: Non-Negative, 1: Negative)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
#from huggingface_hub import notebook_login

# This will prompt your interactive access token pop-up
#notebook_login()

In [9]:
from transformers import TrainingArguments

repo_name = "SamuelCanedo/xlm-roberta-chinese-financial-sentiment"

training_args = TrainingArguments(
    output_dir=repo_name,
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.05,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, # Utilizes mixed-precision tensor cores on T4 GPU
    push_to_hub=False,
)

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Start fine-tuning loop
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.291542,0.246759,0.934000,0.933192
2,0.166391,0.270413,0.942000,0.940189
3,0.130294,0.189614,0.948000,0.946835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=846, training_loss=0.19607548905320765, metrics={'train_runtime': 228.5262, 'train_samples_per_second': 59.061, 'train_steps_per_second': 3.702, 'total_flos': 887719228566120.0, 'train_loss': 0.19607548905320765, 'epoch': 3.0})

In [11]:
# Execute isolated analysis on the testing tier and display metrics
final_evaluation = trainer.evaluate()
print("\n--- Final Test Set Evaluation Metrics ---")
print(final_evaluation)

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.130294,0.189614,3,0.948000,0.946835



--- Final Test Set Evaluation Metrics ---
{'eval_loss': 0.18961358070373535, 'eval_accuracy': 0.948, 'eval_f1': 0.946835484451424}


In [15]:
# Testing of the finetunned model
from transformers import pipeline

#Pipeline
print("Starting the pipeline...")
sentiment_pipeline = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    return_all_scores=True,
    function_to_apply="sigmoid"
)

# Statements for evaluation
test_statements = [
    "该公司因涉嫌严重财务造假被证监会立案调查，股价开盘随即暴跌10%。", # Expected: LABEL_1 (Negativo)
    "最新财报显示，公司第三季度净利润同比增长50%，远超市场预期。",  # Expected: LABEL_0 (Non Negative)
    "中国人民银行今日宣布，维持现行贷款市场报价利率（LPR）保持不变。"   # Expected: LABEL_0 (No Negative)
]

# Label mapping
label_map = {
    "LABEL_0": "NON-NEGATIVE (Neutral/Positive)",
    "LABEL_1": "NEGATIVE"
}

# Run the loop
print("\n--- Results ---")
for i, text in enumerate(test_statements, 1):
    prediction = sentiment_pipeline(text)[0]
    raw_label = prediction["label"]
    confidence = prediction["score"] * 100

    clean_label = label_map.get(raw_label, raw_label)

    print(f"\nTexto {i}: {text}")
    print(f"Inference: {clean_label} | Confidence level: {confidence:.2f}%")

Starting the pipeline...

--- Results ---

Texto 1: 该公司因涉嫌严重财务造假被证监会立案调查，股价开盘随即暴跌10%。
Inference: NEGATIVE | Confidence level: 93.20%

Texto 2: 最新财报显示，公司第三季度净利润同比增长50%，远超市场预期。
Inference: NON-NEGATIVE (Neutral/Positive) | Confidence level: 87.93%

Texto 3: 中国人民银行今日宣布，维持现行贷款市场报价利率（LPR）保持不变。
Inference: NON-NEGATIVE (Neutral/Positive) | Confidence level: 96.25%
